# 02 · Carico e risposta

Come i segnali seguono il carico: quali feature cambiano con il traffico (e rischiano falsi allarmi quando il carico varia) e quali no, quanto conta il mix di richieste, quanto dura il transitorio dopo un cambio di carico.

Fonti: la prova di capacità (`SWEEP`: una scala di rate a mix 0, poi un tratto a 30 req/s per ogni mix) e le sessioni pulite (`DATA`).

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "anomaly" / "build.py").exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anomaly import explore as ex

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 90)

In [ ]:
SWEEP = Path("/home/tecnico/datasets/sweep")
DATA = Path("/home/tecnico/datasets/pilot")
STEP = 30

sweep = {k: ex.load(a, STEP) for k, a in ex.find_runs(SWEEP).items()}
clean = {k: r for k, r in ((k, ex.load(a, STEP)) for k, a in ex.find_runs(DATA).items()) if r.kind == "clean"}
print("prova di capacità:", list(sweep))
print("sessioni pulite:", list(clean))

## Capacità

Rate erogato e p99 di wrk2 per ogni livello della prova di capacità (mix 0).

In [ ]:
for name, run in sweep.items():
    seg = pd.DataFrame(run.labels["segments"]).dropna(subset=["delivered"])
    base = seg[seg.mix == 0].groupby("rate", as_index=False).agg(
        delivered=("delivered", "max"), p99_ms=("p99_ms", "max"), non_2xx=("non_2xx", "sum"))
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(base.rate, base.delivered, "o-", label="erogati")
    ax[0].plot(base.rate, base.rate, "k:", label="richiesti")
    ax[0].set(xlabel="req/s richieste", ylabel="req/s", title=name)
    ax[0].legend()
    ax[1].plot(base.rate, base.p99_ms, "o-")
    ax[1].set(xlabel="req/s", ylabel="p99 ms", ylim=(0, None), title="latenza")
    plt.show()
    display(base)

## Quanto ogni feature segue il rate

Pendenza di log(feature) rispetto a log(rate) sulla scala della prova di capacità, con le mediane per livello a carico stabile. 1 = proporzionale al traffico, 0 = indipendente. Una feature indipendente dal carico che si muove è un buon indizio di guasto; una proporzionale va letta insieme al traffico.

In [ ]:
def elasticity(run, mix=0):
    out = {}
    for feature in run.features:
        levels = ex.by_level(run, feature)
        levels = levels[levels.index.get_level_values("mix") == mix]
        rates = np.log(levels.index.get_level_values("rate").to_numpy(float))
        out[feature] = {}
        for service in run.services:
            y = levels[service].to_numpy(float)
            ok = np.isfinite(y) & (y > 0)
            out[feature][service] = np.polyfit(rates[ok], np.log(y[ok]), 1)[0] if ok.sum() >= 3 else np.nan
    return pd.DataFrame(out)


for name, run in sweep.items():
    e = elasticity(run)
    fig, a = plt.subplots(figsize=(12, 0.3 * len(e) + 1.5))
    image = a.imshow(e.to_numpy(float), cmap="viridis", vmin=0, vmax=1.2, aspect="auto")
    a.set_xticks(range(len(e.columns)), e.columns, rotation=45, ha="right")
    a.set_yticks(range(len(e.index)), e.index, fontsize=8)
    a.set_title(f"{name}: pendenza log-log rispetto al rate")
    fig.colorbar(image)
    plt.show()

## CPU per byte ricevuto

Se il rapporto resta costante tra i livelli, un aumento di CPU senza aumento del traffico in ingresso è sospetto: è il segnale che separa un guasto da un aumento di carico. Millicore per KB/s ricevuto, per i dieci servizi che lavorano di più.

In [ ]:
for name, run in sweep.items():
    cpu, rx = ex.by_level(run, "cpu"), ex.by_level(run, "rx_bytes")
    ratio = (cpu * 1000) / (rx / 1000)
    ratio = ratio[ratio.index.get_level_values("mix") == 0].droplevel("mix").T
    busy = cpu.max().sort_values(ascending=False).index[:10]
    print(name)
    display(ratio.loc[busy].round(2))

## Effetto del mix

A 30 req/s, rapporto tra la mediana con ogni mix e quella con il mix 0, per servizio. Hotel: 1 = più ricerche, 2 = più raccomandazioni, 3 = più prenotazioni. Social: 1 e 2 = più letture di timeline.

In [ ]:
def mix_effect(run, feature, rate):
    levels = ex.by_level(run, feature)
    if rate not in levels.index.get_level_values("rate"):
        return None
    at = levels.xs(rate, level="rate")
    return (at / at.loc[0]).T.drop(columns=0) if 0 in at.index and len(at) > 1 else None


for name, run in sweep.items():
    for feature in ("cpu", "rx_bytes"):
        effect = mix_effect(run, feature, 30)
        if effect is not None:
            print(f"{name}, {feature}")
            display(effect.round(2).sort_values(effect.columns[-1], ascending=False).head(12))

## Transitorio dopo un cambio di carico

Per ogni cambio di rate nelle sessioni pulite: z robusto rispetto ai 2 minuti prima, mediana su servizi ed eventi, separando aumenti e diminuzioni. Dice dopo quanto i segnali si stabilizzano, cioè quanto un detector deve tollerare dopo un cambio.

In [ ]:
BEFORE, AFTER = 120, 360
curves = {"aumento": [], "diminuzione": []}
for name, run in clean.items():
    times, steps = ex.load_changes(run)
    windows, offsets = ex.windows(run, times, BEFORE, AFTER)
    for w, (r0, r1) in zip(windows, steps):
        if r0 != r1:  # i soli cambi di mix sono esclusi
            z = ex.robust_z(w, offsets < 0)
            curves["aumento" if r1 > r0 else "diminuzione"].append(np.nanmedian(z, axis=1))

if any(curves.values()):
    features = next(iter(clean.values())).features
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    for a, (kind, rows) in zip(axes, curves.items()):
        if rows:
            median = np.nanmedian(np.array(rows), axis=0)
            for j, feature in enumerate(features):
                a.plot(offsets, median[:, j], label=feature)
        a.axvline(0, color="k", lw=0.8)
        a.set(title=f"{kind} di rate ({len(rows)} eventi)", xlabel="secondi dal cambio")
    axes[0].set_ylabel("z mediano")
    axes[1].legend(fontsize=7, ncol=2)
    plt.show()
else:
    print("Nessuna sessione pulita con cambi di rate in", DATA)